# Lesson 5: 1000-Cycle ROI Review — Is This Course Worth Your Time?

**The headline number of the AI Agents course, in an executable notebook.**

Duration: 20 min.  Topic: 1000 randomized SMB scenarios, P50 ROI, P05 downside, %positive.

Run the cells top to bottom. No API keys needed. Re-run with different `--n` or `--seed` to test stability.

## Architect Level

- [ ] 🟢 **Junior** (0-2 yr): I can explain what P50 means in plain English.
- [ ] 🟡 **Mid** (2-5 yr): I can read a 1000-cycle report and know which knob to tune.
- [ ] 🟠 **Senior** (5-8 yr): I can re-run the review after every change to the agent and detect regression.
- [ ] 🔴 **Staff** (8+ yr): I can set up the autopilot, diff `latest.json` over time, and write the postmortem.

> **Picking the right level:** Run the next 3 cells. Read the verdict. If you only read one, read the **Verdict** cell output.

## Concept (5 min)

The ROI formula has 6 inputs. Each is uncertain. **The right way to use an uncertain formula is to sample the inputs N times, run the formula, and look at the distribution** — not to plug in a single point estimate and trust it.

This notebook runs the formula **1000 times**, each with the 6 inputs drawn from realistic Asian SMB ranges. The output is a distribution: P05 (the worst 5%), P50 (the median), P95 (the best 5%), and the % of trials where the agent nets positive ROI.

The wrong choice is to estimate the inputs once and ship. The right choice is to **always report the distribution**, not the point estimate, when the inputs are uncertain — which is always.

## Build It: Spec

This notebook must, when run top-to-bottom:
1. Import the formula and the 1000-cycle runner from `code/`
2. Run 1000 trials with `seed=42`
3. Print a banner: `1000 cycles in X.XXs (Y,YYY/s)`
4. Print the stats: N, mean, P05, P25, P50, P75, P95, stddev, %positive, %>=3x
5. Print the best and worst trial
6. Print a **Verdict** line: PASS / MARGINAL / FAIL
7. Save `results/latest.json` and `results/1000_cycle_review_<ts>.md`

**Acceptance criteria:**
- [ ] Notebook executes in < 30 seconds
- [ ] `latest.json` exists after run
- [ ] Markdown report exists after run
- [ ] Verdict is `PASS` under default ranges

In [1]:
# === Standard setup for all L5 notebooks ===
import sys, importlib.util
from pathlib import Path

HERE = Path.cwd()
def find_course_root(p):
    while p != p.parent:
        if (p / 'code').exists() and (p / 'lessons').exists():
            return p
        p = p.parent
    return HERE

COURSE = find_course_root(HERE)
sys.path.insert(0, str(COURSE))

# Load code/run_1000_cycles.py by file path so we don't depend on the
# 'code' package being importable from wherever Jupyter launched.
_mod_path = COURSE / 'code' / 'run_1000_cycles.py'
_spec = importlib.util.spec_from_file_location('run_1000_cycles', _mod_path)
_mod = importlib.util.module_from_spec(_spec)
sys.modules['run_1000_cycles'] = _mod
_spec.loader.exec_module(_mod)
run_cycles = _mod.run_cycles
write_latest_json = _mod.write_latest_json
write_markdown_report = _mod.write_markdown_report
RESULTS_DIR = _mod.RESULTS_DIR

lesson_num = '05'
title = '1000-Cycle ROI Review'
print(f'Setup OK: lesson {lesson_num} = {title}')
print(f'Course root: {COURSE}')
print(f'Results dir: {RESULTS_DIR}')


Setup OK: lesson 05 = 1000-Cycle ROI Review
Course root: /Users/vishnoiprem/PycharmProjects/data-etl-ml/ai-course-public/course/ai-agents
Results dir: /Users/vishnoiprem/PycharmProjects/data-etl-ml/ai-course-public/course/ai-agents/results


In [2]:
# TODO -- implement the 1000-cycle loop
#
# Replace the `pass` below with code that:
#   1. Calls run_cycles(n=1000, seed=42)
#   2. Prints the stats with stats.report()
#   3. Prints a verdict: PASS if pct_positive >= 0.95 else MARGINAL else FAIL
#
# The SOLUTION is in the cell below.  Try it yourself first.

def todo_run_review(n: int = 1000, seed: int = 42) -> None:
    """TODO: implement me."""
    pass

todo_run_review()

In [3]:
# SOLUTION -- run the 1000-cycle review
from datetime import datetime, timezone
import time

N = 1000
SEED = 42
t0 = time.time()
trials, stats, elapsed = run_cycles(N, SEED)
wall = time.time() - t0

print('=' * 60)
print(f'  1000-cycle review  --  {elapsed:.2f}s  ({N/elapsed:,.0f}/s)')
print('=' * 60)
print(stats.report())

# Verdict
print('-' * 60)
if stats.pct_positive >= 0.95 and stats.p05_roi > -100 and stats.median_roi > 0:
    verdict = '**PASS** -- the formula is robust. P50 positive, P05 not catastrophic.'
elif stats.pct_positive >= 0.85 and stats.median_roi > 0:
    verdict = '**MARGINAL** -- positive in most cases but the tail is wide. Tighten inputs.'
else:
    verdict = '**FAIL** -- the formula is not robust under the current ranges.'
print(f'  Verdict: {verdict}')
print('=' * 60)

# Persist
ts = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H-%M-%SZ')
md_path = write_markdown_report(stats, N, SEED, elapsed, ts)
json_path = write_latest_json(stats, N, SEED, elapsed, ts)
print(f'  Wrote: {md_path.name}')
print(f'  Wrote: {json_path.name}')


  1000-cycle review  --  0.00s  (276,305/s)
  N            : 1000
  Mean ROI     : $+335/mo
  Median (P50) : $+239/mo
  P05 (down)   : $+0/mo
  P25          : $+100/mo
  P75          : $+478/mo
  P95 (up)     : $+965/mo
  Stddev       : $328
  % positive   : 95.1%
  % >= 3x pay  : 63.1%
  Best trial   : hrs=5.0/wk  rate=$130/hr  vol=97  adopt=96%  succ=95%  cost=$25/mo  ->  ROI $+2,480/mo (99.3x)
  Worst trial  : hrs=2.0/wk  rate=$39/hr  vol=10  adopt=69%  succ=92%  cost=$97/mo  ->  ROI $-75/mo (-0.8x)

------------------------------------------------------------
  Verdict: **PASS** -- the formula is robust. P50 positive, P05 not catastrophic.
  Wrote: 1000_cycle_review_2026-10-16T02-56-28Z.md
  Wrote: latest.json


In [4]:
# BONUS -- stability sweep across 5 seeds

print('STABILITY SWEEP  --  5 seeds, 1000 cycles each')
print('=' * 70)
print(f'{"seed":<6} {"P05":>10} {"P25":>10} {"P50":>10} {"P75":>10} {"P95":>10} {"%pos":>8} {"%3x":>8}')
print('-' * 70)
for seed in [42, 7, 123, 999, 31415]:
    trials, stats, elapsed = run_cycles(1000, seed)
    print(f'{seed:<6} ${stats.p05_roi:>+8,.0f} ${stats.p25_roi:>+8,.0f} ${stats.median_roi:>+8,.0f} ${stats.p75_roi:>+8,.0f} ${stats.p95_roi:>+8,.0f} {stats.pct_positive:>7.1%} {stats.pct_3x:>7.1%}')
print('=' * 70)
print('  Stable P50 across seeds: the formula is robust, not lucky.')


STABILITY SWEEP  --  5 seeds, 1000 cycles each
seed          P05        P25        P50        P75        P95     %pos      %3x
----------------------------------------------------------------------
42     $      +0 $    +100 $    +239 $    +478 $    +965   95.1%   63.1%
7      $      -4 $     +96 $    +228 $    +466 $  +1,023   94.7%   63.6%
123    $      -4 $    +102 $    +250 $    +517 $  +1,090   94.5%   65.5%
999    $      -7 $     +91 $    +230 $    +475 $  +1,053   94.1%   63.1%
31415  $      -3 $     +99 $    +256 $    +500 $  +1,099   94.4%   66.2%
  Stable P50 across seeds: the formula is robust, not lucky.


## Architect Notes

### Trade-offs

| Choice | Pros | Cons | Pick when |
|---|---|---|---|
| 1 trial per scenario | Fast | Misleading on uncertain inputs | Never, for money decisions |
| 100 trials | Decent | P05 unstable | Quick sanity check |
| **1000 trials (this notebook)** | **P05 stable, %positive accurate** | 30 sec compute | **Standard** |
| 10,000 trials | P05 very stable | 5 min compute, marginal gain | Pre-IPO due diligence |

### Capacity Model

- 1,000 trials × (sample + trial + aggregate) = ~5 ms total on a modern laptop
- Memory: ~1000 floats × 24 bytes = 24 KB
- The bottleneck is disk I/O for `latest.json` (sub-millisecond), not compute

### When NOT to use this lab's content

- **Don't** use uniform sampling if your inputs are correlated (e.g. `adoption_rate` rises with `success_rate`). Use a copula.
- **Don't** use the default ranges if your use case is non-Asian-SMB. Re-fit the ranges from your own data.
- **Don't** ship a 1000-cycle report without reading the **worst** trial. The best trial is marketing; the worst trial is engineering.

### Production Checklist
- [ ] Re-run after every change to the agent (prompt, tool, model, cost)
- [ ] Compare new `latest.json` to old; alert on `pct_positive` drop > 5%
- [ ] Set up monthly cron: re-estimate the 5 inputs from real usage data
- [ ] Archive `results/1000_cycle_review_<ts>.md` to S3 for audit

## Reflect (10 min)

1. Look at the **Worst trial** in the stats output above. Which input is most off? Adjust it and re-run.
2. Look at the **Best trial**. What's the multiple? Is that realistic for your use case?
3. Run the **stability sweep** (cell 7). Are the P50 numbers close across seeds? If yes, the formula is robust. If no, you have a small-sample problem.
4. The `latest.json` is what the autopilot writes every 30 min. Open it in a text editor. That's the entire output of the system.

---

**Read next:**
- `lessons/01-roi-formula.md` -- the lesson that motivates the formula
- `code/agent_roi.py` -- the formula in code (the canonical reference)
- `autopilot/run_autopilot.sh` -- the watchdog that re-runs this forever